In [ ]:
# caminho + supressão de avisos 
import os, warnings
#importação de dados
import pandas as pd
import numpy as np
# conexão com oo banco
from sqlalchemy import create_engine
# aprendizado de maquina
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
# visualização de dados(dashboard)
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
warnings.filterwarnings('ignore')

In [3]:
# conectar ao banco
ENGINE_URL = (
'mysql+pymysql://root:@localhost:3306/bolsa_familia'
)
engine = create_engine(ENGINE_URL, echo=False)

query =  ''' SELECT `MÊS COMPETÊNCIA`, `UF`,
             COUNT(*) AS qtd_parcelas, 
             AVG(`VALOR PARCELA`) AS valor_medio, 
             SUM(`VALOR PARCELA`) AS valor_total 
             FROM `bolsa_familia` 
             GROUP BY `MÊS COMPETÊNCIA`, `UF` '''

df = pd.read_sql(query, engine)

In [ ]:
#EDA
df_uf = df.groupby('UF').agg(media_valor = ('valor_medio', 'mean'),total_parcelas = ('total_parcelas', 'sum')
                             ).reset_index()
print(df_uf)

    UF  media_valor  total_parcelas
0   AC   715.463219          264072
1   AL   675.406654         1071755
2   AM   724.059325         1287026
3   AP   714.428589          244571
4   BA   657.730787         4936765
5   CE   659.207303         2906235
6   DF   664.774635          344166
7   ES   660.234147          614128
8   GO   662.957707          975560
9   MA   692.913321         2460495
10  MG   651.973895         3133196
11  MS   676.810588          398704
12  MT   679.625241          483839
13  PA   694.948774         2684187
14  PB   663.984168         1335605
15  PE   662.033730         3161890
16  PI   665.378618         1182795
17  PR   657.047218         1193432
18  RJ   657.726088         3145779
19  RN   654.652235          998301
20  RO   675.683240          264735
21  RR   734.245384          159027
22  RS   671.862344         1390724
23  SC   659.439484          675207
24  SE   661.881387         1133675
25  SP   657.689810         7324883
26  TO   683.205580         

In [7]:
serie = df_uf['media_valor']
q1, q2, q3 = np.percentile(serie, [25, 50, 75])
iqr = q3 - q1
print(f'\nMedia: {serie.mean():.2f}')
print(f'\nMediana: {serie.median():.2f}')
print(f'\n|Q1: {q1:.2f}|  |Q3: {q3:.2f}| |IQR: {iqr:.2f}|')
print(f'\nAssimetria: {serie.skew():.3f}')
print(f'\nCurtose: {serie.kurt():.3f}')


Media: 675.38

Mediana: 664.77

|Q1: 659.32|  |Q3: 681.42| |IQR: 22.09|

Assimetria: 1.354

Curtose: 0.865


In [ ]:
# aprendizado // agrupamento // não su

features = df_uf[['media_valor', 'total_parcelas']]
# boas praticas para aprendizado de maquina
scaler = StandardScaler()
features_norm = scaler.fit_transform(features)
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df_uf['cluster'] = kmeans.fit_predict(features_norm)
print(df_uf.groupby('cluster')[['media_valor', 'total_parcelas']])